In [1]:
import ast
import os
import json
import requests

import pandas as pd
import dotenv
import redis
import numpy as np

from functools import reduce

In [2]:
# Convert data from byte into datatpyes
def convert_from_byte(byte_dict):
    return {key.decode('utf-8'): value.decode('utf-8') for key, value in byte_dict.items()}

In [3]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [4]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [ ]:
# Get xray data
with open("../src/data/xray_cases.json", "r") as f:
    xray_data = json.load(f)
# get all case_ids from the xray data
case_ids = [case['case_id'] for case in xray_data]

In [ ]:
# Write function to add new participant
def add_participant(participant_id):
    # Check if participant_id is already in the set
    if r.sismember("permitted_ids", participant_id):
        print(f"Participant ID {participant_id} is already in the set.")
        return
    
    # Add participant_id to the set
    r.sadd("permitted_ids", participant_id)
    print(f"Participant ID {participant_id} has been added to the set.")

    # Create hash for the participant with case_ids in randomised order and progress
    randomised_case_ids = np.random.permutation(case_ids).tolist()
    participant_data = {
        "case_ids": json.dumps(randomised_case_ids),
        "progress": 0
    }
    r.hset(f"participant:{participant_id}", mapping=participant_data)
    print(f"Participant data for {participant_id} has been created.")

def delete_participant(participant_id):
    # Check if participant_id is in the set
    if not r.sismember("permitted_ids", participant_id):
        print(f"Participant ID {participant_id} is not in the set.")
        return
    
    # Remove participant_id from the set
    r.srem("permitted_ids", participant_id)
    print(f"Participant ID {participant_id} has been removed from the set.")

    # Delete hash for the participant
    r.delete(f"participant:{participant_id}")
    print(f"Participant data for {participant_id} has been deleted.")


In [ ]:
# Export all submitted responses of a participant to csv
def export_responses(participant_id):
    participant = convert_from_byte(r.hgetall(f"participant:{participant_id}"))
    case_ids = json.loads(participant["case_ids"])

    # Only indices below progress belong to the current enrolment; keys beyond it may be
    # leftovers from an earlier shuffle of the same ID (delete_participant keeps response keys)
    progress = int(participant["progress"])
    rows = []
    for index in range(min(progress, len(case_ids))):
        response = r.hgetall(f"response:{participant_id}:{index}")
        if not response:
            print(f"Warning: response {index} missing for {participant_id}.")
            continue
        response = convert_from_byte(response)
        if response["case_id"] != case_ids[index]:
            print(f"Warning: response {index} is for {response['case_id']}, expected {case_ids[index]}; skipped.")
            continue
        rows.append({"index": index, **response})

    columns = ["index", "case_id"] + [f"item_{i}" for i in range(1, 8)] + ["comment", "duration_ms", "submitted_at"]
    df = pd.DataFrame(rows, columns=columns)
    for column in [f"item_{i}" for i in range(1, 8)] + ["duration_ms"]:
        df[column] = pd.to_numeric(df[column])
    # Flag the second occurrence of the duplicated case (within-rater consistency check)
    df["is_repeat"] = df.duplicated("case_id")

    print(f"{len(df)} of {len(case_ids)} responses found for {participant_id}.")
    df.to_csv(f"{participant_id}_responses.csv", index=False)
    return df

In [ ]:
participant_ids = ["test_participant", "shimul_chan"]

In [10]:
add_participant("test_participant")

Participant ID test_participant has been added to the set.
